# Yambda multi-event reward analysis

Dataset: [yandex/yambda](https://huggingface.co/datasets/yandex/yambda), flat / 50m, `multi_event.parquet`
(all event types in one stream: listen, like, dislike, unlike, undislike), restricted to items that
have audio embeddings (`data/raw/yambda/embeddings.parquet`).

**Interaction definition for the reward:** for a (user, track) pair with at least one listen we call the pair

- **positive** — the track was listened to ≥ 50% of its length at least once (`max played_ratio_pct >= 50`);
- **negative** — the pair has listens, but never ≥ 50%.

This notebook checks how the *other* native event types (like / dislike / unlike / undislike)
distribute across these two groups — the co-occurrence rates are the raw material for the reward model.

**Positive/auxiliary contract.** The datamodule partitions the multi-event stream by
`positive_criterion`: listens played ≥ 50% become the positive training stream (stripped to
user/item/timestamp; they alone define the catalog, k-core, and splits), and everything else —
likes, dislikes, unlikes, undislikes, sub-threshold and unknown-pct listens — lands in
`dm.interaction_store` with its payload columns. The partition happens *before*
`deduplicate_consecutive`, so feedback events can no longer be lost to same-track dedup.

The event-distribution analysis below still uses a separate non-deduplicated canonical frame
restricted to the same catalog: it needs the played ratio of *every* listen (positive ones
included), and the positive stream is deliberately stripped of payload while the store keeps
only non-positive events.

In [ ]:
from __future__ import annotations

import gc
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import flowrec

from flowrec.data.yambda import (
    YambdaDataConfig,
    YambdaDataModule,
    load_yambda_interactions_frame,
)

PROJECT_ROOT = Path(flowrec.__file__).resolve().parent.parent

EMBEDDINGS_PATH = PROJECT_ROOT / "data/raw/yambda/embeddings.parquet"
assert EMBEDDINGS_PATH.exists(), EMBEDDINGS_PATH

TRAIN_WINDOW_LEN: int | None = None
TRAIN_WINDOW_STRIDE: int | None = 25
POSITIVE_PCT_THRESHOLD = 50.0
FEEDBACK_EVENTS = ["like", "dislike", "unlike", "undislike"]

DATASET_PARAMS: dict[str, Any] = {
    "dataset_type": "flat",
    "dataset_size": "50m",
    "event_file": "multi_event",
    "max_history_len": 50,
    "min_user_interactions": 5,
    "min_item_interactions": 5,
    "split_strategy": "leave_one_out",
    "min_history_len": 1,
    "train_example_layout": "dense_causal",
    "train_window_len": TRAIN_WINDOW_LEN,
    "train_window_stride": TRAIN_WINDOW_STRIDE,
    "history_padding": "right",
    "deduplicate_consecutive": False,
    "organic_filter": "any",
    "min_played_ratio_pct": None,
    "positive_criterion": "event_type == 'listen' and played_ratio_pct >= 50",
    "offset_semantic_ids_by_one": False,
    "drop_items_missing_semantic_id": True,
}

# Chart style: dataviz reference palette (validated categorical slots 1-2).
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": INK,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.titlelocation": "left",
    }
)


## 1. Training-view datamodule

The training parameters plus the positive/auxiliary partition: `positive_criterion` selects
listens played ≥ 50% as the positive stream, and the complement is exposed as
`dm.interaction_store`. `embeddings_path` drops every interaction whose track has no audio
embedding (a scope filter — it applies to both streams); no semantic-ID file is passed, so
`drop_items_missing_semantic_id` is a no-op here.
The default `hf_recent_months=2` keeps the last two months relative to the newest event in the file.

In [ ]:
data_cfg = YambdaDataConfig(
    **DATASET_PARAMS,
    use_hf=True,
    embeddings_path=EMBEDDINGS_PATH,
    batch_size=256,
    eval_batch_size=512,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
    include_raw_in_batch=False,
    seed=42,
)
dm = YambdaDataModule(data_cfg)
dm.setup()
print(json.dumps(dm.summary(), indent=2, default=str))


In [ ]:
store = dm.interaction_store
print("auxiliary store:", store.counts_by_type())
print("payload columns:", store.payload_columns)

liked = store.items_for_user("like", 0)
if len(liked):
    agree = bool(store.has_pair("like", np.full(len(liked), 0), liked).all())
    print(f"user 0 liked {len(liked)} catalog tracks; has_pair agrees: {agree}")


In [ ]:
# The deduplicated training frame is not used by this analysis (the frame below
# carries the same events without dedup) -- free it to keep peak memory down.
dm.interactions = pd.DataFrame()
gc.collect()


## 2. Analysis frame: same catalog, no consecutive dedup

Canonical multi-event frame with the embedding filter but **without** `deduplicate_consecutive`
and without the played-ratio floor, restricted to the users and items that survived the
datamodule pipeline (k-core + post-split cold-item filter).

In [ ]:
events = load_yambda_interactions_frame(
    use_hf=True,
    dataset_type=DATASET_PARAMS["dataset_type"],
    dataset_size=DATASET_PARAMS["dataset_size"],
    event_file=DATASET_PARAMS["event_file"],
    allowed_items=dm.embedding_item_ids,
    organic_filter=DATASET_PARAMS["organic_filter"],
    min_played_ratio_pct=None,  # keep all listens; groups are assigned explicitly below
)
n_raw = len(events)
events = events.loc[
    events["raw_user_id"].isin(dm.user2id) & events["raw_item_id"].isin(dm.item2id)
].copy()

# Compact integer pair key instead of string groupby keys.
events["uid"] = events["raw_user_id"].map(dm.user2id).astype("int64")
events["iid"] = events["raw_item_id"].map(dm.item2id).astype("int64")
events["pair"] = events["uid"] * dm.item_vocab_size + events["iid"]
events["event_type"] = events["event_type"].astype("category")
events = events.drop(columns=["raw_user_id", "raw_item_id"])
gc.collect()

print(f"events in window (embedding-filtered): {n_raw:,}")
print(f"events after restricting to the datamodule catalog: {len(events):,}")
print(f"users: {events['uid'].nunique():,}   items: {events['iid'].nunique():,}")


In [ ]:
event_mix = events["event_type"].value_counts().to_frame("events")
event_mix["share_%"] = (100 * event_mix["events"] / len(events)).round(3)
event_mix


## 3. Positive vs negative listen pairs

Aggregate listens to (user, track) pairs, classify by `max played_ratio_pct`, then attach
whether the same user ever produced each feedback event for that track inside the window.

In [ ]:
listens = events.loc[events["event_type"] == "listen"]
pair = listens.groupby("pair", sort=False).agg(
    n_listens=("timestamp", "size"),
    max_pct=("played_ratio_pct", "max"),
    mean_pct=("played_ratio_pct", "mean"),
    organic_share=("is_organic", "mean"),
)
pair["group"] = np.where(pair["max_pct"] >= POSITIVE_PCT_THRESHOLD, "positive", "negative")
pair.loc[pair["max_pct"].isna(), "group"] = "unknown_pct"

for ev in FEEDBACK_EVENTS:
    ev_pairs = events.loc[events["event_type"] == ev, "pair"].unique()
    pair[f"has_{ev}"] = pair.index.isin(ev_pairs)

print(pair["group"].value_counts().to_string())

# Feedback events on tracks the user never listened to inside the window are
# invisible to the pair-level analysis; count them so nothing is silently lost.
listened = pd.Index(pair.index)
orphan = {
    ev: int((~events.loc[events["event_type"] == ev, "pair"].isin(listened)).sum())
    for ev in FEEDBACK_EVENTS
}
print("feedback events without any listen of the same track in the window:", orphan)


In [ ]:
rate_cols = [f"has_{ev}" for ev in FEEDBACK_EVENTS]
group_summary = pair.groupby("group").agg(
    n_pairs=("n_listens", "size"),
    mean_listens=("n_listens", "mean"),
    mean_max_pct=("max_pct", "mean"),
    organic_share=("organic_share", "mean"),
    **{f"{ev}_rate_%": (f"has_{ev}", "mean") for ev in FEEDBACK_EVENTS},
)
for ev in FEEDBACK_EVENTS:
    group_summary[f"{ev}_rate_%"] *= 100
group_summary["share_of_pairs_%"] = 100 * group_summary["n_pairs"] / len(pair)

if {"positive", "negative"} <= set(group_summary.index):
    lift = (
        group_summary.loc["positive", [f"{ev}_rate_%" for ev in FEEDBACK_EVENTS]]
        / group_summary.loc["negative", [f"{ev}_rate_%" for ev in FEEDBACK_EVENTS]]
    )
    print("positive/negative rate lift:")
    print(lift.round(2).to_string())

group_summary.round(3)


## 4. Charts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
vals = listens["played_ratio_pct"].dropna()
ax.hist(vals, bins=50, color=BLUE, edgecolor="white", linewidth=0.4)
ax.axvline(POSITIVE_PCT_THRESHOLD, color=MUTED, linestyle="--", linewidth=1.2)
ax.text(
    POSITIVE_PCT_THRESHOLD + 1.5,
    ax.get_ylim()[1] * 0.95,
    "positive threshold (50%)",
    color=MUTED,
    va="top",
)
ax.set_xlabel("played_ratio_pct of a listen event")
ax.set_ylabel("listen events")
ax.set_title("Played-ratio distribution of listen events (analysis window)")
plt.tight_layout()
plt.show()


In [ ]:
pos = group_summary.loc["positive", [f"{ev}_rate_%" for ev in FEEDBACK_EVENTS]].to_numpy()
neg = group_summary.loc["negative", [f"{ev}_rate_%" for ev in FEEDBACK_EVENTS]].to_numpy()

x = np.arange(len(FEEDBACK_EVENTS))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 4.2))
b1 = ax.bar(x - width / 2, pos, width, label="positive (max pct ≥ 50)", color=BLUE)
b2 = ax.bar(x + width / 2, neg, width, label="negative (max pct < 50)", color=ORANGE)
ax.bar_label(b1, fmt="%.2f%%", padding=2, fontsize=9, color=INK)
ax.bar_label(b2, fmt="%.2f%%", padding=2, fontsize=9, color=INK)
ax.set_xticks(x, FEEDBACK_EVENTS)
ax.set_ylabel("% of (user, track) pairs with the event")
ax.set_title("Feedback co-occurrence by listen group")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# Dose-response: like rate as a function of how much of the track was played.
bins = np.arange(0, 101, 10)
labels = [f"{lo}–{hi}" for lo, hi in zip(bins[:-1], bins[1:])]
known = pair.loc[pair["group"] != "unknown_pct"].copy()
known["pct_bin"] = pd.cut(known["max_pct"], bins=bins, labels=labels, include_lowest=True)
by_bin = known.groupby("pct_bin", observed=True).agg(
    n_pairs=("n_listens", "size"),
    like_rate=("has_like", "mean"),
    dislike_rate=("has_dislike", "mean"),
)
by_bin[["like_rate", "dislike_rate"]] *= 100

fig, ax = plt.subplots(figsize=(8, 4.2))
b = ax.bar(np.arange(len(by_bin)), by_bin["like_rate"], width=0.62, color=BLUE)
ax.bar_label(b, fmt="%.2f%%", padding=2, fontsize=9, color=INK)
ax.set_xticks(np.arange(len(by_bin)), by_bin.index, rotation=0)
ax.set_xlabel("max played_ratio_pct of the pair (bins)")
ax.set_ylabel("% of pairs with a like")
ax.set_title("Like rate vs. how much of the track was played")
plt.tight_layout()
plt.show()

by_bin.round(3)


## 5. Track-level view: how do the positive and negative groups intersect on items?

The groups above are defined per (user, track) pair, so the same track can be positive for one
user and negative for another. Two questions for the reward design:

1. **Overlap** — are negative pairs concentrated on a separate set of "bad" tracks, or do they
   mostly land on tracks that other users listen through?
2. **Feedback profile** — is it true that tracks appearing in negative interactions get few
   likes (and many dislikes) from the listener population as a whole?

If negative-group tracks turn out to be mostly well-liked tracks skipped by *particular* users,
a low played ratio is a personalization signal, not an item-quality signal.

Like/dislike counts below come from users who also listened to the track inside the window
(the pair table), so denominators are consistent; window-orphan feedback stays excluded.

In [ ]:
tp = pair.reset_index()
tp["iid"] = tp["pair"] % dm.item_vocab_size
tp["is_pos"] = tp["group"].eq("positive")
tp["is_neg"] = tp["group"].eq("negative")

track_stats = tp.groupby("iid").agg(
    listeners=("pair", "size"),
    pos_pairs=("is_pos", "sum"),
    neg_pairs=("is_neg", "sum"),
    likes=("has_like", "sum"),
    dislikes=("has_dislike", "sum"),
)

grouped_pairs = track_stats["pos_pairs"] + track_stats["neg_pairs"]
track_stats["pos_share"] = track_stats["pos_pairs"] / grouped_pairs.replace(0, np.nan)

track_stats["track_group"] = np.select(
    [
        (track_stats["pos_pairs"] > 0) & (track_stats["neg_pairs"] == 0),
        (track_stats["neg_pairs"] > 0) & (track_stats["pos_pairs"] == 0),
        (track_stats["pos_pairs"] > 0) & (track_stats["neg_pairs"] > 0),
    ],
    ["only_positive", "only_negative", "mixed"],
    default="unknown_pct_only",
)

n_pos_tracks = int((track_stats["pos_pairs"] > 0).sum())
n_neg_tracks = int((track_stats["neg_pairs"] > 0).sum())
n_both = int(((track_stats["pos_pairs"] > 0) & (track_stats["neg_pairs"] > 0)).sum())
print(f"tracks with >=1 positive pair: {n_pos_tracks:,}")
print(f"tracks with >=1 negative pair: {n_neg_tracks:,}")
print(f"tracks in both sets:           {n_both:,}")
print(f"  = {100 * n_both / n_neg_tracks:.1f}% of negative-set tracks also have positive pairs")
print(f"  = {100 * n_both / n_pos_tracks:.1f}% of positive-set tracks also have negative pairs")

# Where do negative PAIRS land? On tracks that are mostly listened through, or mostly skipped?
neg_pair_track_share = track_stats.loc[track_stats["neg_pairs"] > 0]
neg_weighted = (
    neg_pair_track_share["neg_pairs"]
    .groupby(pd.cut(neg_pair_track_share["pos_share"], [0, 0.25, 0.5, 0.75, 1.0], include_lowest=True))
    .sum()
)
print("\nnegative pairs by the track's positive share:")
print((100 * neg_weighted / neg_weighted.sum()).round(1).to_string())


In [ ]:
profile = track_stats.groupby("track_group").agg(
    n_tracks=("listeners", "size"),
    median_listeners=("listeners", "median"),
    mean_listeners=("listeners", "mean"),
    total_likes=("likes", "sum"),
    total_dislikes=("dislikes", "sum"),
    total_listeners=("listeners", "sum"),
    tracks_with_like=("likes", lambda s: (s > 0).mean()),
    tracks_with_dislike=("dislikes", lambda s: (s > 0).mean()),
)
profile["likes_per_100_listeners"] = 100 * profile["total_likes"] / profile["total_listeners"]
profile["dislikes_per_100_listeners"] = 100 * profile["total_dislikes"] / profile["total_listeners"]
profile["tracks_with_like_%"] = 100 * profile["tracks_with_like"]
profile["tracks_with_dislike_%"] = 100 * profile["tracks_with_dislike"]
profile["share_of_tracks_%"] = 100 * profile["n_tracks"] / len(track_stats)
profile = profile.drop(columns=["tracks_with_like", "tracks_with_dislike"])
profile.round(3)


In [ ]:
# Feedback rates conditioned on how positively the listener population treats the track.
share_bins = [0, 0.25, 0.5, 0.75, 0.999, 1.0]
share_labels = ["0–25%", "25–50%", "50–75%", "75–<100%", "100%"]
graded = track_stats.dropna(subset=["pos_share"]).copy()
graded["share_bin"] = pd.cut(
    graded["pos_share"], share_bins, labels=share_labels, include_lowest=True
)
by_share = graded.groupby("share_bin", observed=True).agg(
    n_tracks=("listeners", "size"),
    total_listeners=("listeners", "sum"),
    total_likes=("likes", "sum"),
    total_dislikes=("dislikes", "sum"),
    zero_like_tracks=("likes", lambda s: (s == 0).mean()),
)
by_share["likes_per_100"] = 100 * by_share["total_likes"] / by_share["total_listeners"]
by_share["dislikes_per_100"] = 100 * by_share["total_dislikes"] / by_share["total_listeners"]
by_share["zero_like_tracks_%"] = 100 * by_share["zero_like_tracks"]

x = np.arange(len(by_share))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 4.2))
b1 = ax.bar(x - width / 2, by_share["likes_per_100"], width, label="likes", color=BLUE)
b2 = ax.bar(x + width / 2, by_share["dislikes_per_100"], width, label="dislikes", color=ORANGE)
ax.bar_label(b1, fmt="%.2f", padding=2, fontsize=9, color=INK)
ax.bar_label(b2, fmt="%.2f", padding=2, fontsize=9, color=INK)
ax.set_xticks(x, by_share.index)
ax.set_xlabel("track's share of positive pairs among its listeners")
ax.set_ylabel("events per 100 listeners")
ax.set_title("Track-level feedback vs. how positively the audience listens")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

by_share[["n_tracks", "total_listeners", "likes_per_100", "dislikes_per_100", "zero_like_tracks_%"]].round(3)


## 6. Per-user view: conditional feedback rates for each user

Sections 3–5 pooled all users together. Here every user gets their own conditional rates:

- `p_like_positive` — of this user's **positive** pairs, the share that also carries their like;
- `p_like_negative` — the same conditioned on their **negative** pairs;
- likewise for dislike / unlike / undislike.

Pooled rates can be driven by a small hyper-active minority, so the per-user distribution tells
us whether "likes concentrate in positives" is a property of *most users* or of a few. That
decides whether a like-based reward term generalizes across users or needs per-user calibration.

In [ ]:
up = pair.loc[pair["group"] != "unknown_pct"].reset_index()
up["uid"] = up["pair"] // dm.item_vocab_size

agg = up.groupby(["uid", "group"], observed=True).agg(
    n_pairs=("pair", "size"),
    **{ev: (f"has_{ev}", "sum") for ev in FEEDBACK_EVENTS},
)
counts = agg.unstack("group")

user_stats = pd.DataFrame(index=counts.index)
for grp in ["positive", "negative"]:
    user_stats[f"n_{grp}"] = counts[("n_pairs", grp)].fillna(0).astype(int)
    for ev in FEEDBACK_EVENTS:
        user_stats[f"{ev}_{grp}"] = counts[(ev, grp)].fillna(0).astype(int)
        user_stats[f"p_{ev}_{grp}"] = (
            user_stats[f"{ev}_{grp}"] / user_stats[f"n_{grp}"].replace(0, np.nan)
        )
user_stats["n_pairs_total"] = user_stats["n_positive"] + user_stats["n_negative"]
user_stats["pos_share"] = user_stats["n_positive"] / user_stats["n_pairs_total"]

print(f"users: {len(user_stats):,}")
print(f"pairs per user: mean {user_stats['n_pairs_total'].mean():.0f}, "
      f"median {user_stats['n_pairs_total'].median():.0f}")
print(f"users with >=1 positive pair: {(user_stats['n_positive'] > 0).mean() * 100:.1f}%, "
      f">=1 negative pair: {(user_stats['n_negative'] > 0).mean() * 100:.1f}%")
print(f"per-user positive share: mean {user_stats['pos_share'].mean():.3f}, "
      f"median {user_stats['pos_share'].median():.3f}")


In [ ]:
rows = []
for ev in FEEDBACK_EVENTS:
    for grp in ["positive", "negative"]:
        r = user_stats[f"p_{ev}_{grp}"].dropna()
        rows.append(
            {
                "event": ev,
                "group": grp,
                "users_with_group": len(r),
                "mean_user_rate_%": 100 * r.mean(),
                "median_user_rate_%": 100 * r.median(),
                "p90_user_rate_%": 100 * r.quantile(0.9),
                "users_with_zero_rate_%": 100 * r.eq(0).mean(),
                "pooled_rate_%": group_summary.loc[grp, f"{ev}_rate_%"],
            }
        )
per_user_summary = pd.DataFrame(rows).set_index(["event", "group"])
per_user_summary.round(3)


In [ ]:
# Is the direction of the signal consistent user by user?
both = user_stats.loc[(user_stats["n_positive"] > 0) & (user_stats["n_negative"] > 0)]

likers = both.loc[(both["like_positive"] + both["like_negative"]) > 0]
frac_like_users = 100 * len(likers) / len(both)
like_pos_gt = 100 * (likers["p_like_positive"] > likers["p_like_negative"]).mean()
like_neg_gt = 100 * (likers["p_like_positive"] < likers["p_like_negative"]).mean()
print(f"users with both groups: {len(both):,}")
print(f"  of them ever like a listened track: {frac_like_users:.1f}%")
print(f"  among likers: P(like|pos) > P(like|neg) for {like_pos_gt:.1f}% of users, "
      f"< for {like_neg_gt:.1f}% (rest equal)")

dislikers = both.loc[(both["dislike_positive"] + both["dislike_negative"]) > 0]
frac_dis_users = 100 * len(dislikers) / len(both)
dis_neg_gt = 100 * (dislikers["p_dislike_negative"] > dislikers["p_dislike_positive"]).mean()
dis_pos_gt = 100 * (dislikers["p_dislike_negative"] < dislikers["p_dislike_positive"]).mean()
print(f"  ever dislike a listened track: {frac_dis_users:.1f}%")
print(f"  among dislikers: P(dislike|neg) > P(dislike|pos) for {dis_neg_gt:.1f}% of users, "
      f"< for {dis_pos_gt:.1f}%")


In [ ]:
MIN_PAIRS = 20
pos_rates = 100 * user_stats.loc[user_stats["n_positive"] >= MIN_PAIRS, "p_like_positive"]
neg_rates = 100 * user_stats.loc[user_stats["n_negative"] >= MIN_PAIRS, "p_like_negative"]

bins = np.arange(0, 25.5, 1.0)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.hist(
    pos_rates.clip(upper=25), bins=bins,
    weights=np.full(len(pos_rates), 100 / len(pos_rates)),
    color=BLUE, alpha=0.75, label=f"P(like | positive), n={len(pos_rates):,}",
)
ax.hist(
    neg_rates.clip(upper=25), bins=bins,
    weights=np.full(len(neg_rates), 100 / len(neg_rates)),
    color=ORANGE, alpha=0.6, label=f"P(like | negative), n={len(neg_rates):,}",
)
ax.set_xlabel(f"per-user like rate, % (users with ≥{MIN_PAIRS} pairs in the group; clipped at 25)")
ax.set_ylabel("% of users")
ax.set_title("Distribution of per-user like rates, positive vs negative pairs")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(
    100 * user_stats["pos_share"].dropna(),
    bins=np.arange(0, 101, 2.5),
    weights=np.full(user_stats["pos_share"].notna().sum(), 100 / user_stats["pos_share"].notna().sum()),
    color=BLUE, edgecolor="white", linewidth=0.3,
)
ax.axvline(100 * user_stats["pos_share"].median(), color=MUTED, linestyle="--", linewidth=1.2)
ax.text(
    100 * user_stats["pos_share"].median() - 2, ax.get_ylim()[1] * 0.95,
    f"median {100 * user_stats['pos_share'].median():.0f}%", color=MUTED, ha="right", va="top",
)
ax.set_xlabel("share of the user's pairs that are positive, %")
ax.set_ylabel("% of users")
ax.set_title("How completion behavior varies across users")
plt.tight_layout()
plt.show()


## Observations

*(numbers from the positive/auxiliary-contract run: `positive_criterion` = listens ≥ 50%,
catalog defined by positives only)*

**Dataset (training view).** 8,796 users, 134,802 tracks, 7.24M positive interactions after
k-core, dedup, and the embedding filter. The auxiliary store holds 4.19M events with payload:
3.94M sub-threshold/unknown-pct listens, 196k likes, 31.8k unlikes, 18.5k dislikes, 2.3k
undislikes (plus 490k auxiliary events dropped because their user/item did not survive the
positive-stream k-core — counted in the store's build stats). Event mix in the analysis window:
97.8% listens, 1.72% likes, 0.28% unlikes, 0.16% dislikes, 0.02% undislikes.

**The played-ratio distribution is strongly bimodal.** Of 4.70M (user, track) listen pairs,
65% sit in the 90–100% bin and 21% in the 0–10% bin; the middle is a valley. The 50% threshold
therefore separates two behavioral modes almost perfectly: positive pairs have mean max
played ratio 98.0, negative pairs 9.7. Users either listen through a track or skip it early.
Groups: 3.27M positive pairs (69.7%) vs 1.42M negative (30.3%).

**Likes concentrate in the positive group.** 4.76% of positive pairs also carry a like vs
1.76% of negative pairs — a 2.7× lift. The dose-response is monotone: like rate grows from
1.27% (0–10% played) to 5.17% (80–90%), flattening at 4.77% for 90–100%. A listen ≥ 50% is a
meaningful engagement signal, and a like on top of it is a reliable strong-positive marker.

**Dislikes invert.** 0.73% of negative pairs have a dislike vs 0.22% of positive pairs
(3.4× more likely in the negative group), and the dislike rate collapses to 0.17% in the
90–100% bin. Short listens are not just "no interaction" — they carry active-rejection signal,
which argues for treating the low-played group as a soft negative rather than pure absence.

**unlike / undislike are rare and nearly group-independent** (unlike: 0.44% vs 0.52%;
undislike: ≈0.03% both). On their own they add little; their main use is reconstructing the
final like/dislike state of a pair rather than serving as reward events.

**Bookkeeping.** 8.6k like events and 8.9k unlikes reference tracks with no listen by that
user inside the window; they are excluded from the pair-level rates, so a like-based reward
term should decide explicitly whether to count them. Organic share is slightly higher among
negative pairs (37.8% vs 33.2%).

**Reward implication.** A first reward can be: base positive for max played ratio ≥ 50%,
a bonus for a co-occurring like (strong positive, rate lift 2.7×), and a penalty for a
dislike (concentrated in the < 50% group). The monotone like-rate curve also supports a
graded (non-binary) engagement term over played ratio.

### Track-level findings (section 5)

**The negative group is not a set of "bad tracks".** Under the positive-defined catalog this
holds *by construction* — every catalog track has ≥ 5 positive listens, so 100% of tracks with
negative pairs also have positive ones (the earlier criterion-less run established the same
point empirically at 97.6%). Weighting by pairs: 89.2% of all negative pairs land on tracks
where the *majority* of listeners are positive (pos share > 0.5), and only 0.3% land on
predominantly skipped tracks (pos share ≤ 0.25). Skips are personal/contextual, not item quality.

**Likes do not separate the groups at the item level.** Likes per 100 listeners sit in a
narrow 2.5–4.8 band across positive-share bins and are non-monotone. The hypothesis "tracks
from the negative group have few likes" is not supported.

**Dislike rate is the one clean item-level signal.** It falls monotonically with positive
share: 0.78 per 100 listeners for skip-heavy tracks (0–25%) down to 0.15 for fully-positive
tracks (≈5×).

**Reward implication.** The played-ratio-based negative should be treated as a *personalization*
signal (this user × this track), not propagated to the item. If an item-quality prior is wanted,
build it from the track's dislike rate (or its positive-listen share), not from like counts.

### Per-user findings (section 6)

**Coverage.** All 8,796 catalog users have at least one positive pair (by construction of the
positive-defined catalog); median 419 pairs per user, so per-user rates are well estimated.

**Completion behavior varies a lot between users.** The per-user positive share spans nearly
the full range (median 73%, mean 69%), with a long tail of skip-heavy users. A reward built on
raw played ratio inherits this user effect; per-user centering is worth considering.

**The like signal is broad and consistently directional.** 71.8% of users like at least one
listened track. Among them, **91.9% individually have P(like | positive) > P(like | negative)**
— the association is not a minority artifact. Per-user P(like | positive): mean 6.4%,
median 1.6%, p90 17.9%; 31% of users never like a positive pair. The per-user mean (6.4%)
exceeds the pooled rate (4.8%), i.e. lighter listeners like relatively more often.

**The dislike signal is sparse at the user level.** Only 30.6% of users ever dislike a listened
track, and among them the direction P(dislike | negative) > P(dislike | positive) holds for
just 57.5% of users individually — per-user dislike estimates are too rare to be reliable,
even though the pooled direction (0.73% vs 0.22%) is clear. A reward should use dislikes as
pooled/item-level evidence or a strong per-event penalty, not as a per-user rate.

**undislike/unlike are negligible per user** (≈91–97% of users have a zero rate in either group).

**Reward implication.** A like-bonus term generalizes across users but its magnitude is heavily
user-dependent (median 1.6% vs p90 17.9%) — consider normalizing the like bonus by the user's
own like propensity. The dislike penalty should not be personalized; apply it per event.